<a href="https://colab.research.google.com/github/almagomen/v2v/blob/main/colab_launch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video2Video – Despliegue en Colab

1. **Activa GPU**: *Runtime → Change runtime type → T4 GPU*
2. Ejecuta las celdas en orden.
3. Configura **Secrets** (panel izquierdo 🔑): `GOOGLE_API_KEY`, `LMNT_API_KEY`

## 1. Clonar repo e instalar dependencias

In [ ]:
import os
import sys
import importlib
import warnings
from google.colab import drive, userdata
warnings.filterwarnings("ignore", category=UserWarning)

# Gestión del Repositorio (Clone o Pull)
repo_path = "/content/v2v"
get_ghp = userdata.get('GHP_GITHUB_TOKEN')
repo_url = f"https://{get_ghp}@github.com/almagomen/v2v.git"

if not os.path.exists(repo_path):
    !git clone -q {repo_url} {repo_path}
    %cd {repo_path}
else:
    %cd {repo_path}
    !git pull

# Instalación de Dependencias (Solo la primera vez o si faltan)
if 'deps_installed' not in globals():
    print("📦 Instalando dependencias...")
    !uv pip install "yt-dlp[default]" bs-roformer-infer -q
    !uv pip install -U git+https://github.com/jianfch/stable-ts.git --no-deps -q
    !uv pip install openai-whisper --no-deps -q
    !uv pip install git+https://github.com/pyannote/pyannote-audio.git --no-deps -q
    !uv pip install 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q
    !uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' sentencex 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' 'torch-audiomentations>=0.12.0' torchcodec==0.7.0 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q
    deps_installed = True

# Recarga de módulos (Hot-Reload)
def reload_project():
    paquetes_locales = ['src', 'app_gradio']
    for mod_name in list(sys.modules.keys()):
        if any(mod_name.startswith(pkg) for pkg in paquetes_locales):
            try: importlib.reload(sys.modules[mod_name])
            except Exception: del sys.modules[mod_name]

if 'initial_setup_done' in globals():
    print("\n♻️ Recargando módulos...")
    reload_project()

# Gestión de Gradio e Imports
import gradio
gradio.close_all()
from src.stt import get_whisper_model
from src.diarization import get_diarization_model
from src.download import get_roformer_model
from src.config import QUEUE_DIR, TEMP_DIR
from app_gradio import demo

# Carga de modelos y Lanzamiento
if 'initial_setup_done' not in globals():
    print("🛠️  Cargando modelos...")
    drive.mount("/content/drive")
    get_whisper_model()
    get_diarization_model()
    get_roformer_model()

initial_setup_done = True

print("\n🚀 Lanzando Gradio...")
demo.launch(
    share=True,
    allowed_paths=[str(QUEUE_DIR), str(TEMP_DIR)],
    inline=False,
    debug=True
)